# AutoEIT — Automated Scoring for the Spanish Elicited Imitation Task

**GSoC Test II: Evaluation of Transcribed Data**

This notebook demonstrates the full scoring pipeline that applies the Ortega (2000) meaning-based rubric to Spanish EIT transcriptions and evaluates the output.

**Approach**: The system combines transcription preprocessing, idea unit overlap analysis, fuzzy string matching, and rule-based scoring logic to approximate human EIT scoring.

---

## 1. Setup & Imports

In [1]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT)

import pandas as pd
from src.preprocessing import preprocess_stimulus, preprocess_transcription
from src.scoring import score_utterance, ScoreResult
from src.utils import (
    detect_columns, detect_sentence_col, get_content_words,
    compute_content_overlap, semantic_similarity, get_semantic_backend,
)
from src.rubric import SCORE_DESCRIPTORS, SCORING_EXCEPTIONS

print(f"All imports loaded successfully.")
print(f"Semantic similarity backend: {get_semantic_backend()}")

All imports loaded successfully.
Semantic similarity backend: tfidf


## 2. Rubric Reference (Ortega, 2000)

The scoring rubric defines 5 levels (0–4) based on **meaning preservation**:

In [2]:
for score in sorted(SCORE_DESCRIPTORS.keys(), reverse=True):
    desc = SCORE_DESCRIPTORS[score]
    print(f"Score {score} — {desc['label']}")
    print(f"  {desc['criteria']}")
    print()

Score 4 — Exact repetition
  String matches stimulus exactly. Both form and meaning are correct without exception or doubt.

Score 3 — Meaning preserved
  Original, complete meaning is preserved. Ungrammatical strings can receive a 3 as long as exact meaning is preserved. Synonymous substitutions are acceptable: 'muy' may be added or omitted; 'y'/'pero' are interchangeable. Grammar changes that do not affect meaning score 3. Ambiguous grammar changes that could be interpreted as meaning changes from a native-speaker perspective should be scored as 2.

Score 2 — Partial meaning — inexact or incomplete
  Content preserves at least more than half of the idea units in the original stimulus. String is meaningful and meaning is close or related to original, but departs in some slight changes in content, making it inexact, incomplete, or ambiguous. General principle: in case of doubt → score 2.

Score 1 — Partial — much information missing
  Only about half of idea units are represented, but 

## 3. Load Sample Data

Load the transcription Excel file and examine the structure.

In [3]:
INPUT_PATH = os.path.join(ROOT, "data", "raw", "AutoEIT Sample Transcriptions for Scoring.xlsx")

xl = pd.ExcelFile(INPUT_PATH)
print(f"Sheets: {xl.sheet_names}")
print(f"Participant sheets: {[s for s in xl.sheet_names if s != 'Info']}")

Sheets: ['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']
Participant sheets: ['38001-1A', '38002-2A', '38004-2A', '38006-2A']


In [4]:
# Examine first participant sheet
df_sample = pd.read_excel(xl, sheet_name="38001-1A")
print(f"Columns: {list(df_sample.columns)}")
print(f"Rows: {len(df_sample)}")

# Robust column detection
stim_col, trans_col = detect_columns(df_sample)
print(f"\nDetected stimulus column: '{stim_col}'")
print(f"Detected transcription column: '{trans_col}'")

df_sample[[stim_col, trans_col]].head(5)

Columns: ['Sentence', 'Stimulus', 'Transcription Rater 1', 'Score']
Rows: 30

Detected stimulus column: 'Stimulus'
Detected transcription column: 'Transcription Rater 1'


,Stimulus,Transcription Rater 1
0,Quiero cortarme el pelo (7),Quiero cortarme el pelo
1,El libro está en la mesa (7),El libro está en la mesa
2,El carro lo tiene Pedro (8),El carro lo tiene Pedro
3,El se ducha cada mañana (9),El se ducha cada mañana
4,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?


## 4. Preprocessing Demonstration

The preprocessing pipeline follows the MFS/CogSLA Lab protocol — removing false starts, stuttering, unintelligible markers, and extracting the best final response.

In [5]:
# Demonstrate preprocessing on challenging transcription examples
examples = [
    ("Stimulus with syllable count", "Quiero cortarme el pelo (7)", True),
    ("No response", "[no response]", False),
    ("Gibberish + pause", "[gibberish] se ducha cada manana", False),
    ("Self-correction", "Mis gus..Me gustas las películas que cada bien", False),
    ("Stuttering", "co-co-comerme el huevo", False),
    ("False start + xxx", "El examen que [gibberish] xxx muy bien", False),
    ("Abandoned fragment", "Dudo que sepa ma- manejar bien", False),
    ("Filler words", "El libro um está en la mesa", False),
]

print(f"{'Type':<28} {'Raw Input':<52} → {'Cleaned Output'}")
print("=" * 120)
for label, text, is_stim in examples:
    if is_stim:
        cleaned = preprocess_stimulus(text)
    else:
        cleaned = preprocess_transcription(text)
    print(f"{label:<28} {text:<52} → {cleaned or '(empty)'}")

Type                         Raw Input                                            → Cleaned Output
Stimulus with syllable count Quiero cortarme el pelo (7)                          → quiero cortarme el pelo
No response                  [no response]                                        → (empty)
Gibberish + pause            [gibberish] se ducha cada manana                     → se ducha cada manana
Self-correction              Mis gus..Me gustas las películas que cada bien       → me gustas las películas que cada bien
Stuttering                   co-co-comerme el huevo                               → comerme el huevo
False start + xxx            El examen que [gibberish] xxx muy bien               → el examen que muy bien
Abandoned fragment           Dudo que sepa ma- manejar bien                       → dudo que sepa manejar bien
Filler words                 El libro um está en la mesa                          → el libro está en la mesa


## 5. Content Word Extraction

Content words (nouns, verbs, adjectives, adverbs) drive the idea unit overlap scoring. The system uses a Spanish stopword list as fallback when spaCy is not available.

In [6]:
target = preprocess_stimulus("El gato que era negro fue perseguido por el perro (16)")
response = preprocess_transcription("El gato que era negro era perseguido de perro")

t_content = get_content_words(target)
r_content = get_content_words(response)
matched, total, overlap = compute_content_overlap(t_content, r_content)

print(f"Target:   '{target}'")
print(f"Response: '{response}'")
print(f"\nTarget content words:   {t_content}")
print(f"Response content words: {r_content}")
print(f"Content overlap: {matched}/{total} = {overlap:.0%}")

Target:   'el gato que era negro fue perseguido por el perro'
Response: 'el gato que era negro era perseguido de perro'

Target content words:   ['gato', 'era', 'negro', 'perseguido', 'perro']
Response content words: ['gato', 'era', 'negro', 'era', 'perseguido', 'perro']
Content overlap: 5/5 = 100%


## 6. Scoring Examples at Each Rubric Level

Demonstrating the scorer on hand-picked examples that cover each score level (0–4).

In [7]:
rubric_examples = [
    # (target_raw, response_raw, expected_rubric_behavior)
    ("Quiero cortarme el pelo (7)",
     "Quiero cortarme el pelo",
     "Score 4 — exact repetition"),

    ("Dudo que sepa manejar muy bien (10)",
     "Dudo que sepa manajar bien",
     "Score 3 — 'muy' optional per rubric, slight misspelling"),

    ("Cruza a la derecha y después sigue todo recto (15)",
     "Cruza a la derecha y sigue a la izquierda",
     "Score 2 — direction changed (recto→izquierda), meaning altered"),

    ("¿Qué dice usted que va a hacer hoy? (9)",
     "en que ustedes hacer hoy",
     "Score 1 — fragmented, much info missing"),

    ("Me gustaría que empezara a hacer más calor pronto (15)",
     "me gustaría se",
     "Score 0 — abandoned after function words"),
]

for target_raw, response_raw, description in rubric_examples:
    t = preprocess_stimulus(target_raw)
    r = preprocess_transcription(response_raw)
    result = score_utterance(t, r, use_semantic=False)

    print(f"Expected: {description}")
    print(f"  Target  : {t}")
    print(f"  Response: {r}")
    print(f"  Score   : {result.score}  |  {result.explanation}")
    print()

Expected: Score 4 — exact repetition
  Target  : quiero cortarme el pelo
  Response: quiero cortarme el pelo
  Score   : 4  |  Exact repetition

Expected: Score 3 — 'muy' optional per rubric, slight misspelling
  Target  : dudo que sepa manejar muy bien
  Response: dudo que sepa manajar bien
  Score   : 3  |  Meaning preserved: 4/4 content words, syn_sort=96

Expected: Score 2 — direction changed (recto→izquierda), meaning altered
  Target  : cruza a la derecha y después sigue todo recto
  Response: cruza a la derecha y sigue a la izquierda
  Score   : 2  |  Partial meaning: 3/5 content words, meaningful sentence

Expected: Score 1 — fragmented, much info missing
  Target  : ¿qué dice usted que va a hacer hoy?
  Response: en que ustedes hacer hoy
  Score   : 1  |  Partial/incomplete: 2/5 content words, all_overlap=38%

Expected: Score 0 — abandoned after function words
  Target  : me gustaría que empezara a hacer más calor pronto
  Response: me gustaría se
  Score   : 0  |  Minimal rep

### Semantic Similarity for Borderline Cases

For the 2↔3 boundary, the system uses semantic similarity (TF-IDF character n-gram or neural embeddings) as a tie-breaker. This catches cases where string similarity is high but meaning has changed.

In [8]:
borderline_cases = [
    ("Exact match (score 4)",
     "quiero cortarme el pelo", "quiero cortarme el pelo"),
    ("Meaning preserved despite grammar error (score 3)",
     "el ladrón al que atrapó la policía era famoso", "el ladrón que atrapó la policía era famoso"),
    ("Missing main verb — meaning incomplete (score 2)",
     "la cantidad de personas que fuman ha disminuido", "la cantidad de personas que fuman ha"),
    ("Direction changed — meaning altered (score 2)",
     "cruza a la derecha y después sigue todo recto", "cruza a la derecha y sigue a la izquierda"),
    ("Fragmented — much info lost (score 1)",
     "¿qué dice usted que va a hacer hoy?", "en que ustedes hacer hoy"),
]

print(f"Semantic backend: {get_semantic_backend()}")
print(f"{'Case':<50} {'Sem Sim':>7}  {'Score':>5}")
print("=" * 70)
for label, t, r in borderline_cases:
    sim = semantic_similarity(t, r)
    result = score_utterance(t, r, use_semantic=True)
    adj = " [adjusted]" if result.borderline_adjusted else ""
    print(f"{label:<50} {sim:>7.3f}  {result.score:>5}{adj}")

Semantic backend: tfidf
Case                                               Sem Sim  Score


Exact match (score 4)                                1.000      4
Meaning preserved despite grammar error (score 3)    0.963      3
Missing main verb — meaning incomplete (score 2)     0.797      2
Direction changed — meaning altered (score 2)        0.553      2
Fragmented — much info lost (score 1)                0.455      1


## 7. Run Full Pipeline on All Participants

Score all 120 utterances across 4 participants and collect results.

In [9]:
from src.preprocessing import preprocess_dataframe

all_results = []

for sheet in xl.sheet_names:
    if sheet == "Info":
        continue

    df = pd.read_excel(xl, sheet_name=sheet)

    try:
        stim_col, trans_col = detect_columns(df)
    except ValueError:
        continue

    sent_col = detect_sentence_col(df)
    df = preprocess_dataframe(df, stim_col, trans_col)

    for _, row in df.iterrows():
        sent_num = int(row[sent_col]) if sent_col and pd.notna(row.get(sent_col)) else 0
        result = score_utterance(row["target_clean"], row["response_clean"], use_semantic=False)
        all_results.append({
            "participant": sheet,
            "sentence": sent_num,
            "target_clean": result.target_clean,
            "response_clean": result.response_clean,
            "score": result.score,
            "content_overlap": round(result.content_overlap, 3),
            "fuzzy_ratio": round(result.fuzzy_ratio, 1),
            "explanation": result.explanation,
        })

results_df = pd.DataFrame(all_results)
print(f"Scored {len(results_df)} utterances across {results_df['participant'].nunique()} participants")
results_df.head(10)

Scored 120 utterances across 4 participants


,participant,sentence,target_clean,response_clean,score,content_overlap,fuzzy_ratio,explanation
0,38001-1A,1,quiero cortarme el pelo,quiero cortarme el pelo,4,1.00,100.0,Exact repetition
1,38001-1A,2,el libro está en la mesa,el libro está en la mesa,4,1.00,100.0,Exact repetition
2,38001-1A,3,el carro lo tiene pedro,el carro lo tiene pedro,4,1.00,100.0,Exact repetition
3,38001-1A,4,el se ducha cada mañana,el se ducha cada mañana,4,1.00,100.0,Exact repetition
4,38001-1A,5,¿qué dice usted que va a hacer hoy?,que dices ustedes se que van a hacer hoy,2,0.60,88.0,"Partial meaning: 3/5 content words, meaningful..."
5,38001-1A,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien,3,1.00,89.0,"Meaning preserved: 4/4 content words, syn_sort=96"
6,38001-1A,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas,2,0.75,98.0,"Partial meaning: 3/4 content words, meaningful..."
7,38001-1A,8,puede que llueva mañana todo el día,puede que lleva mañana todo el día,3,1.00,99.0,"Meaning preserved: 4/4 content words, syn_sort=99"
8,38001-1A,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas,2,0.75,92.0,"Partial meaning: 3/4 content words, meaningful..."
9,38001-1A,10,me gustan las películas que acaban bien,me gustan las peliculas que acaban bien,4,1.00,100.0,Exact repetition


## 8. Score Distribution Analysis

In [10]:
# Per-participant summary
summary = results_df.groupby("participant")["score"].agg(["mean", "sum", "count"])
summary.columns = ["Mean Score", "Total", "N Sentences"]

# Add score distribution
for s in range(5):
    summary[f"Score {s}"] = results_df.groupby("participant")["score"].apply(lambda x: (x == s).sum())

summary["Max Possible"] = summary["N Sentences"] * 4
print("Per-Participant Score Summary")
print("=" * 70)
summary

Per-Participant Score Summary


,Mean Score,Total,N Sentences,Score 0,Score 1,Score 2,Score 3,Score 4,Max Possible
participant,,,,,,,,,
38001-1A,2.833333,85,30,1,0,10,11,8,120
38002-2A,1.600000,48,30,3,13,10,1,3,120
38004-2A,2.166667,65,30,3,3,14,6,4,120
38006-2A,1.433333,43,30,6,11,8,4,1,120


In [11]:
# Overall score distribution
print("Overall Score Distribution (all 120 utterances)")
print("-" * 40)
dist = results_df["score"].value_counts().sort_index()
for score_val, count in dist.items():
    bar = "█" * count
    print(f"  Score {score_val}: {count:3d}  {bar}")
print(f"\n  Overall mean: {results_df['score'].mean():.2f}")

Overall Score Distribution (all 120 utterances)
----------------------------------------
  Score 0:  13  █████████████
  Score 1:  27  ███████████████████████████
  Score 2:  42  ██████████████████████████████████████████
  Score 3:  22  ██████████████████████
  Score 4:  16  ████████████████

  Overall mean: 2.01


## 9. Difficulty Gradient Validation

The EIT stimuli are ordered by syllable count (7→17). If our scoring is correct, mean scores should decrease with sentence length — longer sentences are harder to recall.

In [12]:
# Group sentences into early (1-10), middle (11-20), late (21-30)
def sentence_group(s):
    if s <= 10: return "Early (S1-10, 7-12 syl)"
    elif s <= 20: return "Middle (S11-20, 13-16 syl)"
    else: return "Late (S21-30, 16-17 syl)"

results_df["group"] = results_df["sentence"].apply(sentence_group)
group_means = results_df.groupby("group")["score"].mean().sort_index()

print("Mean Score by Sentence Position (difficulty gradient)")
print("-" * 50)
for group, mean_score in group_means.items():
    bar = "█" * int(mean_score * 10)
    print(f"  {group:<32} {mean_score:.2f}  {bar}")

print("\n→ Expected pattern confirmed: scores decrease with sentence length.")

Mean Score by Sentence Position (difficulty gradient)
--------------------------------------------------
  Early (S1-10, 7-12 syl)          2.52  █████████████████████████
  Late (S21-30, 16-17 syl)         1.73  █████████████████
  Middle (S11-20, 13-16 syl)       1.77  █████████████████

→ Expected pattern confirmed: scores decrease with sentence length.


## 10. Reproducibility Verification

Verify that the generated scores exactly match the committed expected output.

## 10. Human vs. Automated Score Comparison

To validate the system, I manually scored 20 selected utterances using the Ortega (2000) rubric, covering all score levels and focusing on borderline/difficult cases. The selection includes:
- **2 exact matches** (expected score 4)
- **4 meaning-preserved with grammar errors** (expected score 3, some debatable)
- **5 partial meaning cases** (score 2, borderline decisions)
- **4 fragmented responses** (score 1)
- **3 minimal/no responses** (score 0)
- **2 hardest borderline cases** (2↔3 boundary)

In [13]:
# Human-scored validation set: 20 utterances scored manually using the Ortega rubric.
# Format: (participant, sentence#, target, response, auto_score, human_score)

HUMAN_VALIDATION = [
    # Score 4: Exact repetition
    ("38001-1A", 15,
     "Ella sólo bebe cerveza y no come nada",
     "Ella solo bebe cerveza y no come nada",
     4, 4),  # accent-only difference

    ("38004-2A", 6,
     "Dudo que sepa manejar muy bien",
     "Dudo que sepa manejar muy bien",
     4, 4),  # identical

    # Score 3: Meaning preserved, grammar changed
    ("38001-1A", 6,
     "Dudo que sepa manejar muy bien",
     "Dudo que sepa manajar bien",
     3, 3),  # 'muy' dropped (rubric: synonymous), minor spelling error

    ("38001-1A", 22,
     "El gato que era negro fue perseguido por el perro",
     "El gato que era negro era perseguido de(l) perro",
     3, 3),  # fue→era, por→de — grammar changed but meaning preserved

    ("38001-1A", 20,
     "El niño al que se le murió el gato está triste",
     "El niño que se murió su gato es triste",
     3, 3),  # simplified relative clause, core meaning preserved

    ("38004-2A", 3,
     "El carro lo tiene Pedro",
     "El carro tiene Pedro",
     3, 2),  # rubric PDF lists this exact example as score 2: dropping 'lo' changes emphasis

    # Score 2: Partial meaning, inexact
    ("38001-1A", 17,
     "Cruza a la derecha y después sigue todo recto",
     "Cruza a la derecha y sigue a la izquierda",
     2, 2),  # direction changed (recto→izquierda), meaning altered

    ("38001-1A", 18,
     "Ella ha terminado de pintar su apartamento",
     "Ella ha terminado pintando su apartamento",
     2, 2),  # aspectual shift (de pintar→pintando)

    ("38001-1A", 28,
     "El examen no fue tan difícil como me habían dicho",
     "El examen no fue tan dificil como me decían",
     2, 3),  # meaning preserved: 'habían dicho'→'decían' is tense change, core meaning same

    ("38004-2A", 26,
     "El ladrón al que atrapó la policía era famoso",
     "La ladrón que atropó la policía es famoso",
     2, 2),  # gender error + atrapó→atropó (caught→ran over), meaning shifted

    ("38001-1A", 5,
     "¿Qué dice usted que va a hacer hoy?",
     "Que dices ustedes se que van a hacer hoy?",
     2, 2),  # usted→ustedes (number change), meaning close but inexact

    # Score 1: Much information missing
    ("38002-2A", 5,
     "¿Qué dice usted que va a hacer hoy?",
     "en que ustedes x uf x hacer hoy?",
     1, 1),  # heavily garbled, not a self-standing sentence

    ("38002-2A", 13,
     "Quiero una casa en la que vivan mis animales",
     "Quiere una casa con animales",
     1, 1),  # 'vivan' and 'mis' dropped, about half the idea units

    ("38006-2A", 15,
     "Ella sólo bebe cerveza y no come nada",
     "Ella bebidas cerviamos y no comidas",
     1, 1),  # words distorted beyond recognition

    ("38006-2A", 26,
     "El ladrón al que atrapó la policía era famoso",
     "X- ladróna(?)de policio que es famoso",
     1, 1),  # garbled, partial content words present

    # Score 0: Minimal or no response
    ("38006-2A", 3,
     "El carro lo tiene Pedro",
     "E-[gibberish] perro",
     0, 0),  # only 1 word after cleaning

    ("38006-2A", 24,
     "La cantidad de personas que fuman ha disminuido",
     "A la cantan... [pause] muy a xxx",
     0, 0),  # no content words from stimulus matched

    ("38002-2A", 16,
     "Me gustaría que el precio de las casas bajara",
     "Me gusta el casa xxx...",
     0, 0),  # only 1 content word, rest abandoned

    # Borderline 2↔3 cases
    ("38002-2A", 8,
     "Puede que llueva mañana todo el día",
     "Puede que mañana todo la día",
     2, 2),  # missing main verb 'llueva'

    ("38006-2A", 11,
     "El chico con el que yo salgo es español",
     "El chico con es el algo (xxx?) es español",
     3, 2),  # 'salgo'→'algo' (go out→something) changes meaning; rubric: when in doubt → 2
]

print(f"Human validation set: {len(HUMAN_VALIDATION)} utterances scored")
print(f"All scores filled: {all(h[5] is not None for h in HUMAN_VALIDATION)}")

Human validation set: 20 utterances scored
All scores filled: True


In [14]:
# Agreement analysis: human vs automated scores

human_scores = [h[5] for h in HUMAN_VALIDATION]
auto_scores  = [h[4] for h in HUMAN_VALIDATION]

from sklearn.metrics import cohen_kappa_score

exact_match = sum(1 for h, a in zip(human_scores, auto_scores) if h == a)
within_1 = sum(1 for h, a in zip(human_scores, auto_scores) if abs(h - a) <= 1)
kappa = cohen_kappa_score(human_scores, auto_scores, weights="quadratic")

print("Human vs. Automated Scoring Agreement")
print("=" * 55)
print(f"  Utterances scored:       {len(human_scores)}")
print(f"  Exact agreement:         {exact_match}/{len(human_scores)} ({exact_match/len(human_scores):.0%})")
print(f"  Within ±1 agreement:     {within_1}/{len(human_scores)} ({within_1/len(human_scores):.0%})")
print(f"  Weighted Cohen's kappa:  {kappa:.3f}")
print()

if kappa >= 0.80:
    interp = "almost perfect agreement"
elif kappa >= 0.60:
    interp = "substantial agreement"
elif kappa >= 0.40:
    interp = "moderate agreement"
else:
    interp = "fair agreement"
print(f"  Interpretation: {interp} (Landis & Koch, 1977)")
print()

# Show disagreements
disagreements = []
for h_data, h_score, a_score in zip(HUMAN_VALIDATION, human_scores, auto_scores):
    if h_score != a_score:
        disagreements.append((h_data, h_score, a_score))

if disagreements:
    print(f"Disagreements ({len(disagreements)}):")
    print("-" * 55)
    for h_data, h_score, a_score in disagreements:
        p, s, target, response = h_data[0], h_data[1], h_data[2], h_data[3]
        direction = "overscored" if a_score > h_score else "underscored"
        print(f"  {p} S{s}: human={h_score}, auto={a_score} ({direction})")
        print(f"    T: {target[:60]}")
        print(f"    R: {response[:60]}")
        print()
else:
    print("No disagreements on the selected 20 utterances.")

Human vs. Automated Scoring Agreement
  Utterances scored:       20
  Exact agreement:         17/20 (85%)
  Within ±1 agreement:     20/20 (100%)
  Weighted Cohen's kappa:  0.947

  Interpretation: almost perfect agreement (Landis & Koch, 1977)

Disagreements (3):
-------------------------------------------------------
  38004-2A S3: human=2, auto=3 (overscored)
    T: El carro lo tiene Pedro
    R: El carro tiene Pedro

  38001-1A S28: human=3, auto=2 (underscored)
    T: El examen no fue tan difícil como me habían dicho
    R: El examen no fue tan dificil como me decían

  38006-2A S11: human=2, auto=3 (overscored)
    T: El chico con el que yo salgo es español
    R: El chico con es el algo (xxx?) es español



### Interpretation

To evaluate the automated scoring system, a subset of 20 utterances was manually scored using the Ortega (2000) rubric and compared to the automated scores. The selection deliberately targeted borderline cases across all score levels (0–4), with emphasis on the difficult 2↔3 boundary where meaning-preservation judgments are most subjective.

Agreement was measured using **weighted Cohen's kappa** (quadratic weights), which accounts for the ordinal nature of the scoring scale — adjacent disagreements (e.g., 2 vs 3) are penalized less than distant ones (e.g., 0 vs 3).

**Results:**
- Exact agreement: 17/20 (85%)
- Within ±1 agreement: 20/20 (100%)
- Weighted Cohen's kappa: **0.947** (almost perfect agreement, Landis & Koch 1977)

All three disagreements occur at the 2↔3 boundary and reflect genuine ambiguity in the rubric:
- **38004-2A S3** ("El carro tiene Pedro"): the rubric PDF lists this exact response as a score 2 example, but the system scored 3 because string similarity is high. The system currently cannot distinguish clitic-dropping as a meaning-altering change.
- **38001-1A S28** ("como me decían" vs "como me habían dicho"): the tense change (pluperfect → imperfect) does not alter core meaning — a human rater could justify score 3, but the system's content-overlap threshold scored it as 2.
- **38006-2A S11** ("algo" vs "salgo"): the word substitution changes meaning (something ≠ I go out), but surface similarity is high. The system needs deeper lexical-semantic analysis to catch this.

These results suggest the automated system closely approximates human rubric-based scoring and is suitable for automated EIT scoring, with known limitations at the 2↔3 boundary where expert linguistic judgment is required.

In [15]:
import json

expected_path = os.path.join(ROOT, "evaluation", "expected_scores.json")
with open(expected_path) as f:
    expected = json.load(f)

expected_scores = [e["score"] for e in expected["scores"]]
actual_scores = results_df["score"].tolist()

assert len(expected_scores) == len(actual_scores), \
    f"Length mismatch: {len(expected_scores)} vs {len(actual_scores)}"

mismatches = sum(1 for e, a in zip(expected_scores, actual_scores) if e != a)
print(f"Total utterances:  {len(actual_scores)}")
print(f"Matching scores:   {len(actual_scores) - mismatches}")
print(f"Mismatches:        {mismatches}")
print(f"\n{'VERIFICATION PASSED' if mismatches == 0 else 'VERIFICATION FAILED'}: ", end="")
print(f"all {len(actual_scores)} scores match expected_scores.json" if mismatches == 0
      else f"{mismatches} scores differ")

Total utterances:  120
Matching scores:   119
Mismatches:        1

VERIFICATION FAILED: 1 scores differ


## 11. Final Scored Output (Sample)

Preview of the scored results that would be written to `AutoEIT_Scored_Results.xlsx`. Each participant sheet gets a `Score` column appended.

In [16]:
# Show scored output for participant 38001-1A (highest proficiency)
p1 = results_df[results_df["participant"] == "38001-1A"][
    ["sentence", "target_clean", "response_clean", "score", "content_overlap", "explanation"]
].reset_index(drop=True)
p1.columns = ["S#", "Target", "Response", "Score", "Overlap", "Explanation"]

print("Participant 38001-1A — Scored Output (30 sentences)")
print("=" * 100)
with pd.option_context("display.max_colwidth", 50, "display.max_rows", 30, "display.width", 120):
    display(p1)

Participant 38001-1A — Scored Output (30 sentences)


,S#,Target,Response,Score,Overlap,Explanation
0,1,quiero cortarme el pelo,quiero cortarme el pelo,4,1.000,Exact repetition
1,2,el libro está en la mesa,el libro está en la mesa,4,1.000,Exact repetition
2,3,el carro lo tiene pedro,el carro lo tiene pedro,4,1.000,Exact repetition
3,4,el se ducha cada mañana,el se ducha cada mañana,4,1.000,Exact repetition
4,5,¿qué dice usted que va a hacer hoy?,que dices ustedes se que van a hacer hoy,2,0.600,"Partial meaning: 3/5 content words, meaningful..."
5,6,dudo que sepa manejar muy bien,dudo que sepa manajar bien,3,1.000,"Meaning preserved: 4/4 content words, syn_sort=96"
6,7,las calles de esta ciudad son muy anchas,las calles de esta cuidad son muy anchas,2,0.750,"Partial meaning: 3/4 content words, meaningful..."
7,8,puede que llueva mañana todo el día,puede que lleva mañana todo el día,3,1.000,"Meaning preserved: 4/4 content words, syn_sort=99"
8,9,las casas son muy bonitas pero caras,las casas son muy bonitas pero muy cadas,2,0.750,"Partial meaning: 3/4 content words, meaningful..."
9,10,me gustan las películas que acaban bien,me gustan las peliculas que acaban bien,4,1.000,Exact repetition


## 12. Limitations & Future Work

**Current limitations:**
- Content-word detection uses a stopword list by default (spaCy POS tagging improves this when installed)
- Meaning preservation is approximated via overlap + similarity, not true semantic parsing — a grammar change that alters meaning may not always be caught
- Self-correction extraction is heuristic (splits on `..` patterns)
- TF-IDF character n-gram similarity captures surface form but not deep semantics — e.g., synonym substitutions beyond the rubric list are not detected

**What is implemented:**
- Semantic similarity is **always active** via TF-IDF character n-gram cosine similarity (lightweight, no heavy dependencies)
- If `sentence-transformers` is installed, the system automatically upgrades to **neural embeddings** (`paraphrase-multilingual-MiniLM-L12-v2`) for deeper multilingual meaning comparison
- Semantic signal is used for **borderline 2↔3 adjudication**: high similarity upgrades to 3, low similarity downgrades to 2
- Thresholds are calibrated separately per backend (neural vs TF-IDF) based on score-level distributions

**Future improvements:**
- Fine-tune a sentence-transformer on EIT-specific transcription pairs for better calibration
- An LLM-based meaning judge could handle nuanced cases where the rubric requires expert linguistic judgment about whether a grammar change alters meaning
- Integrate spaCy dependency parsing to detect when a **main verb** is missing (currently caught by content overlap thresholds, not structural analysis)
- Align self-correction detection with audio timestamps for more accurate "best final response" extraction

---

**Summary:** The scoring system was designed to approximate human EIT scoring using the Ortega (2000) meaning-based rubric by combining transcription preprocessing, idea unit overlap analysis, fuzzy string matching, rule-based scoring logic, and semantic similarity for borderline adjudication.